# Multi-Agent Financial Analysis System

AAI-520 Final Team Project

## 1. Project Overview and GitHub Repository

Describe the project goals, team members, and repository link.

## 2. Setup and Configuration

Import dependencies, load environment settings, and define reproducibility controls.

## 3. Agent Design and Shared State

Explain the agent roles, shared workflow state, and orchestration graph.

## 4. Agent Functions, Data Sources, and Tool Use

Demonstrate research planning, isolated provider adapters beginning with Yahoo Finance, and dynamic use of market, financial, and news tools.

### News data source (mine)

For news I'm pulling from NewsAPI.org instead of a static file. I give it a company name or ticker and it hands back recent articles with a title, publisher, date, url, and body text. Since it's a live pull, what comes back changes every time I run this, which is the point, we want current news.

Market data (Yahoo Finance) and financial statement data are Person 1 and Person 2's pieces. Those tool functions are still empty stubs as of this run, so I'm not demoing them here. I'll drop a note lower in this notebook once they're ready.

In [1]:
import sys
from pathlib import Path

# so we can import the src package from inside notebooks/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

from src.tools.news_tools import get_company_news

TICKER = "AAPL"
articles = get_company_news(TICKER, company_name="Apple")

print(f"Pulled {len(articles)} deduped articles for {TICKER}")
for a in articles[:3]:
    print(f"- {a['title']} ({a['publisher']}, {a['date']})")

Pulled 10 deduped articles for AAPL
- Apple M6チップ搭載のMac miniのCPUベンチマークは、Single-CoreスコアはApple Siliconの中で最も高く、Multi-CoreスコアでもM1 Ultraに迫るスコアに。 (Applech2.com, 2026-09-21T22:59:20Z)
- Apple AirPods Pro 3 $169 + 2 Years AppleCare+ — Costco via DoorDash [YMMV] (Slickdeals.net, 2026-09-21T22:57:42Z)
- Apple Seeds Second macOS Golden Gate 27.2 Beta to Developers for Testing (Mactrast.com, 2026-09-21T22:38:58Z)


## 5. Workflow 1 — Prompt Chaining

Demonstrate: Ingest News → Preprocess → Classify → Extract → Summarize.

I built this as five separate functions, ingest, preprocess, classify, extract, summarize, so each stage's output is visible on its own instead of one black box call. First two stages don't need an LLM at all, they're just plain Python.

In [2]:
from src.workflows.news_pipeline import ingest, preprocess

ingested = ingest(articles)
preprocessed = preprocess(ingested)

print(f"Ingested: {len(ingested)} articles")
print(f"After preprocess (deduped, cleaned): {len(preprocessed)} articles")
if preprocessed:
    print("\nSample cleaned article:")
    print(preprocessed[0]["title"])
    print(preprocessed[0]["text"][:200])

Ingested: 10 articles
After preprocess (deduped, cleaned): 10 articles

Sample cleaned article:
Apple M6チップ搭載のMac miniのCPUベンチマークは、Single-CoreスコアはApple Siliconの中で最も高く、Multi-CoreスコアでもM1 Ultraに迫るスコアに。
Apple M6Mac miniCPUSingle-CoreApple SiliconMulti-CoreM1 Ultra Apple20260922Apple2Apple M6M5 ProMac mini (2026)Mac mini (M6/M5 Pro)CPUGeekbench Browser MacMac mini (M6)Mac mini (M5 Pro)Mac Studio (M… [


Classify, extract, and summarize are the three stages that actually call an LLM. I haven't locked in which model we're using yet, `OPENAI_MODEL` is blank in `.env.example` and nobody's picked a provider. I asked the team about doing this with a local model instead of paying for OpenAI, still waiting to hear back.

So this cell will run without crashing (I built in a fallback so a missing key doesn't break the chain), but right now the category comes back as OTHER and sentiment as neutral for everything, those are placeholder defaults, not a real classification. Once we settle on a provider and I drop a key into `.env`, rerunning this cell gives the real output. Leaving it in so the shape of the pipeline is visible either way.

In [3]:
from src.workflows.news_pipeline import classify, extract, summarize

classified = classify(preprocessed)
extracted = extract(classified)
news_summary = summarize(TICKER, extracted)

for a in classified[:3]:
    print(f"- [{a['category']}] {a['title']}")

print("\nSummary:")
print(news_summary)

- [OTHER] Apple M6チップ搭載のMac miniのCPUベンチマークは、Single-CoreスコアはApple Siliconの中で最も高く、Multi-CoreスコアでもM1 Ultraに迫るスコアに。
- [OTHER] Apple AirPods Pro 3 $169 + 2 Years AppleCare+ — Costco via DoorDash [YMMV]
- [OTHER] Apple Seeds Second macOS Golden Gate 27.2 Beta to Developers for Testing

Summary:
Unable to generate a news summary for AAPL.


## 6. Workflow 2 — Routing

Demonstrate how content is routed to the appropriate specialist agents.

This one's Person 1's. Router is still an empty stub, nothing to demo here yet.

## 7. Workflow 3 — Evaluator–Optimizer

Show the initial analysis, quality evaluation, feedback, and refined analysis.

This one's Person 2's. Evaluator and Optimizer are still empty stubs, nothing to demo here yet.

## 8. Memory and Learning Across Runs

Demonstrate how concise lessons are stored and retrieved for a later run.

Memory is stored as JSON, keyed by ticker, path comes from `MEMORY_PATH` in `.env`. Each entry gets a timestamp so we can tell a fresh note from a stale one. This part doesn't need an LLM or any external API, so I can run and verify it fully right now.

Below I save a note for AAPL like the kind an Evaluator would leave after a weak run, then load it back to show the round trip actually works on disk.

In [4]:
from src.memory.memory_store import save_memory, load_memory

save_memory(
    TICKER,
    feedback="Run 1: news coverage was thin, only checked one source window. Widen the lookback next time.",
    research_topics=["news_coverage_depth"],
)

memories = load_memory(TICKER)
print(f"{len(memories)} memory entries for {TICKER}")
for m in memories:
    print(f"[{m['timestamp']}] {m['feedback']}")

2 memory entries for AAPL
[2026-09-22T23:02:36.838127+00:00] Run 1: news coverage was thin, only checked one source window. Widen the lookback next time.
[2026-09-22T23:01:29.766942+00:00] Run 1: news coverage was thin, only checked one source window. Widen the lookback next time.


## 9. End-to-End Investment Research Example

Run the complete system for a selected stock symbol and present its sourced report.

Can't run the full thing yet. Planner, Router, Market Agent, Financial Agent, Evaluator, Optimizer, and Synthesis Agent are all still empty stubs, and `graph.py` hasn't been wired up. My News Agent (`run_news_agent`) is done on its own and returns sourced news intelligence, but it needs the rest of the graph to actually show up in an end-to-end run.

Picking this section back up once Person 1 and Person 2's agents land and we've settled on an LLM provider.

## 10. Evaluation, Limitations, and Conclusions

Summarize evaluation results, known limitations, responsible-use considerations, and future improvements.